# Exercise 2.1: Bike-Sharing Demand Prediction with PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_1_bike_sharing_mlp_pytorch.ipynb)

This exercise trains a small neural network on a real urban mobility dataset.

We use the UCI Bike Sharing Dataset. Each row describes one hour in the Capital Bikeshare system, including time, weather, and calendar variables. The model predicts the total number of bike rentals in that hour.

You will practice:

- downloading a real tabular dataset
- preparing time and weather features
- scaling inputs and the target variable
- building a regression neural network with `torch.nn.Module`
- training with `MSELoss`, backpropagation, and `Adam`
- evaluating predictions with MAE, RMSE, and R2


## 0. Colab setup

Run this first in Colab. The dataset is downloaded directly from UCI in the next section.


In [ ]:
!pip -q install torch scikit-learn pandas matplotlib seaborn

In [ ]:
import io
import random
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Download the UCI Bike Sharing dataset

The hourly file contains 17,379 records from 2011 and 2012. The target variable is `cnt`, the total number of bike rentals in one hour.

We do not use `casual` or `registered` as features because they add up to the target and would leak the answer.


In [ ]:
bike_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"

with urllib.request.urlopen(bike_url) as response:
    zip_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    with zf.open("hour.csv") as f:
        bike_df = pd.read_csv(f)

season_map = {1: "winter", 2: "spring", 3: "summer", 4: "fall"}
weather_map = {
    1: "clear / partly cloudy",
    2: "mist / cloudy",
    3: "light snow or rain",
    4: "heavy rain or snow",
}

bike_df["season_name"] = bike_df["season"].map(season_map)
bike_df["weather_name"] = bike_df["weathersit"].map(weather_map)

print("Rows, columns:", bike_df.shape)
display(bike_df.head())
print(bike_df[["dteday", "hr", "season_name", "weather_name", "temp", "hum", "windspeed", "cnt"]].head())


## 2. Inspect demand patterns

Bike demand is strongly related to hour of day, working day, season, and weather. These patterns are visible before training a model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

hourly_mean = bike_df.groupby(["hr", "workingday"], as_index=False)["cnt"].mean()
sns.lineplot(data=hourly_mean, x="hr", y="cnt", hue="workingday", marker="o", ax=axes[0])
axes[0].set_title("Average rentals by hour")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Average rentals")
axes[0].legend(title="Working day")

sns.boxplot(data=bike_df, x="season_name", y="cnt", order=["winter", "spring", "summer", "fall"], ax=axes[1])
axes[1].set_title("Rental counts by season")
axes[1].set_xlabel("")
axes[1].set_ylabel("Hourly rentals")

plt.tight_layout()
plt.show()


### Student task 1: Read the plots

Answer briefly:

- What hours have the highest demand on working days?
- What hours have high demand on non-working days?
- Which season tends to have lower demand?

<details>
<summary>Answer / tip</summary>

On working days, demand usually peaks around commuting times: morning and late afternoon. On non-working days, demand is often more spread through the daytime.

Winter tends to have lower demand. This is plausible because weather and daylight affect cycling.
</details>


## 3. Prepare features for a neural network

Neural networks need numeric inputs. We use:

- continuous weather variables: `temp`, `atemp`, `hum`, `windspeed`
- binary or numeric calendar variables: `yr`, `holiday`, `workingday`
- cyclical encodings for hour, month, and weekday
- one-hot encodings for season and weather situation

The target `cnt` is also scaled. Training on scaled counts is easier than training directly on raw rental counts.


In [ ]:
bike_features = bike_df.copy()

bike_features["hr_sin"] = np.sin(2 * np.pi * bike_features["hr"] / 24)
bike_features["hr_cos"] = np.cos(2 * np.pi * bike_features["hr"] / 24)
bike_features["mnth_sin"] = np.sin(2 * np.pi * bike_features["mnth"] / 12)
bike_features["mnth_cos"] = np.cos(2 * np.pi * bike_features["mnth"] / 12)
bike_features["weekday_sin"] = np.sin(2 * np.pi * bike_features["weekday"] / 7)
bike_features["weekday_cos"] = np.cos(2 * np.pi * bike_features["weekday"] / 7)

numeric_features = [
    "yr",
    "holiday",
    "workingday",
    "temp",
    "atemp",
    "hum",
    "windspeed",
    "hr_sin",
    "hr_cos",
    "mnth_sin",
    "mnth_cos",
    "weekday_sin",
    "weekday_cos",
]

categorical_features = ["season", "weathersit"]
feature_df = pd.get_dummies(
    bike_features[numeric_features + categorical_features],
    columns=categorical_features,
    prefix=categorical_features,
    drop_first=False,
).astype("float32")

target = bike_features["cnt"].to_numpy(dtype="float32").reshape(-1, 1)

print("Feature count:", feature_df.shape[1])
display(feature_df.head())


## 4. Train, validation, and test split

We use a random split for this introductory exercise. For a real forecasting project, a time-based split is usually more realistic because the model should predict future dates from past data.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    feature_df.values,
    target,
    test_size=0.30,
    random_state=SEED,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled = y_scaler.transform(y_val)
y_test_scaled = y_scaler.transform(y_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

print("Train / val / test:", len(train_ds), len(val_ds), len(test_ds))
xb, yb = next(iter(train_loader))
print("X batch:", xb.shape, "y batch:", yb.shape)


## 5. Define a small regression neural network

The model is a multi-layer perceptron (MLP). It outputs one number: the predicted bike rental count, in scaled target units.


In [ ]:
class BikeDemandMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.network(x)

model = BikeDemandMLP(input_dim=feature_df.shape[1], hidden_dim=64).to(DEVICE)
print(model)


In [ ]:
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    sample_pred = model(xb[:4].to(DEVICE))

print("Input shape: ", xb[:4].shape)
print("Output shape:", sample_pred.shape)
print("First predictions in scaled units:", sample_pred.squeeze().cpu().numpy().round(3))


## 6. Train the network

The training loop is the same workflow as in the lecture:

1. forward pass
2. compute loss
3. backpropagation with `loss.backward()`
4. parameter update with `optimizer.step()`

For regression, we use mean squared error loss.


In [ ]:
def inverse_count(y_scaled):
    return y_scaler.inverse_transform(y_scaled.reshape(-1, 1)).ravel()


def evaluate(model, data_loader, loss_fn):
    model.eval()
    losses = []
    preds_scaled = []
    targets_scaled = []

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            losses.append(loss.item() * len(xb))
            preds_scaled.append(pred.cpu().numpy())
            targets_scaled.append(yb.cpu().numpy())

    preds_scaled = np.vstack(preds_scaled).ravel()
    targets_scaled = np.vstack(targets_scaled).ravel()
    preds = np.clip(inverse_count(preds_scaled), 0, None)
    targets = inverse_count(targets_scaled)

    mse = mean_squared_error(targets, preds)
    return {
        "loss": sum(losses) / len(data_loader.dataset),
        "mae": mean_absolute_error(targets, preds),
        "rmse": np.sqrt(mse),
        "r2": r2_score(targets, preds),
    }

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 120
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

    train_metrics = evaluate(model, train_loader, loss_fn)
    val_metrics = evaluate(model, val_loader, loss_fn)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "train_rmse": train_metrics["rmse"],
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_r2": val_metrics["r2"],
        }
    )

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch {epoch:03d} | "
            f"train loss {train_metrics['loss']:.3f} | val loss {val_metrics['loss']:.3f} | "
            f"val MAE {val_metrics['mae']:.1f} | val RMSE {val_metrics['rmse']:.1f} | val R2 {val_metrics['r2']:.3f}"
        )

history_df = pd.DataFrame(history)


## 7. Plot learning curves

The loss is computed on scaled rental counts. RMSE is shown in the original unit: number of bikes rented in one hour.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", label="train", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="val_loss", label="validation", ax=axes[0])
axes[0].set_title("MSE loss during training")
axes[0].set_ylabel("Scaled MSE loss")

sns.lineplot(data=history_df, x="epoch", y="train_rmse", label="train", ax=axes[1])
sns.lineplot(data=history_df, x="epoch", y="val_rmse", label="validation", ax=axes[1])
axes[1].set_title("RMSE during training")
axes[1].set_ylabel("RMSE: rentals per hour")

plt.tight_layout()
plt.show()

display(history_df.tail())


### Student task 2: Change training settings

Change one setting and rerun the training cells:

- `hidden_dim`: try `16`, `64`, and `128`
- `lr`: try `0.0003`, `0.001`, and `0.003`
- `EPOCHS`: try `40`, `120`, and `200`

What changes in the validation RMSE curve?

<details>
<summary>Answer / tip</summary>

A small learning rate usually trains more slowly. A larger hidden layer can reduce training error, but it may not always improve validation RMSE.

For this dataset, compare settings with `val_rmse` and `val_mae`. A lower training loss alone is not enough if validation error gets worse.
</details>


## 8. Evaluate on the test set

Use the test set only after you have chosen model settings. The metrics below are in the original rental-count units.


In [ ]:
test_metrics = evaluate(model, test_loader, loss_fn)
print(f"Test MAE:  {test_metrics['mae']:.1f} rentals/hour")
print(f"Test RMSE: {test_metrics['rmse']:.1f} rentals/hour")
print(f"Test R2:   {test_metrics['r2']:.3f}")


In [ ]:
model.eval()
with torch.no_grad():
    test_pred_scaled = model(X_test_tensor.to(DEVICE)).cpu().numpy().ravel()

test_pred = np.clip(inverse_count(test_pred_scaled), 0, None)
test_true = y_test.ravel()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(test_true, test_pred, alpha=0.35, s=18)
limit = max(test_true.max(), test_pred.max())
ax.plot([0, limit], [0, limit], color="black", linewidth=2)
ax.set_xlabel("Actual rentals")
ax.set_ylabel("Predicted rentals")
ax.set_title("Test predictions")
plt.show()


## 9. Predict one observed day

The next cell predicts the last 24 hours in the dataset and compares actual and predicted bike rentals.


In [ ]:
example_rows = bike_df.tail(24).copy()
example_features = feature_df.tail(24).values
example_scaled = x_scaler.transform(example_features)
example_tensor = torch.tensor(example_scaled, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    example_pred_scaled = model(example_tensor).cpu().numpy().ravel()

example_rows["predicted_cnt"] = np.clip(inverse_count(example_pred_scaled), 0, None)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(example_rows["hr"], example_rows["cnt"], marker="o", label="actual")
ax.plot(example_rows["hr"], example_rows["predicted_cnt"], marker="o", label="predicted")
ax.set_title(f"Observed day: {example_rows['dteday'].iloc[0]}")
ax.set_xlabel("Hour")
ax.set_ylabel("Bike rentals")
ax.legend()
plt.show()

display(example_rows[["dteday", "hr", "temp", "hum", "windspeed", "cnt", "predicted_cnt"]].head())


## 10. Short reflection

Answer these questions in two or three sentences each:

- What are the inputs, hidden layers, output value, loss, and optimizer in this notebook?
- Why do we scale both features and the target variable?
- Why are MAE and RMSE easier to interpret than scaled MSE loss?
- Why would a time-based split be important for a real forecasting deployment?

The point is to understand the training workflow: forward pass, loss, backpropagation, parameter update, and evaluation on unseen data.
